## **NLP**

In [1]:
!pip install datasets nltk scikit-learn pandas matplotlib seaborn

Load IMDb Dataset IMDB Dataset of 50K Movie Reviews


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


Convert to DataFrame

In [8]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
dataset = load_dataset(
    'csv',
    data_files=f'{path}/IMDB Dataset.csv'
)

# Convert to dataframe
df = pd.DataFrame(dataset['train'])

# Convert sentiment to labels
df['label'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

# Split dataset
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

Generating train split: 0 examples [00:00, ? examples/s]

Display Number of Samples

In [9]:
print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

Training samples: 40000
Testing samples: 10000


Class Distribution

In [10]:
print(train_df['label'].value_counts())

label
1    20000
0    20000
Name: count, dtype: int64


Print 5 Positive Reviews

In [11]:
positive_reviews = train_df[train_df['label'] == 1]

for i in range(5):
    print("\nPositive Review", i+1)
    print(positive_reviews.iloc[i]['review'])


Positive Review 1
I caught this little gem totally by accident back in 1980 or '81. I was at a revival theatre to see two old silly sci-fi movies. The theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). Most were somewhat amusing but THIS came on and, within seconds, the audience was in hysterics! The biggest laugh came when they showed "Princess Laia" having huge cinnamon buns instead of hair on her head. She looks at the camera, gives a grim smile and nods. That made it even funnier! You gotta see "Chewabacca" played by what looks like a Muppet! It was extremely silly and stupid...but I couldn't stop laughing. Most of the dialogue was drowned out because of all the laughter. Also if you know "Star Wars" pretty well it's even funnier--they deliberately poke fun at some of the dialogue. This REALLY works with an audience! A definite 10!

Positive Review 2
Opera (the U.S. title is terror at the opera) is somewhat of a letdow

Print 5 Negative Reviews

In [12]:
negative_reviews = train_df[train_df['label'] == 0]

for i in range(5):
    print("\nNegative Review", i+1)
    print(negative_reviews.iloc[i]['review'])


Negative Review 1
I can't believe that I let myself into this movie to accomplish a favor my friends ask me early this April 14, 2007. This movie certainly a pain in your ass in theater and sickly boring, I haven't even felt the gory impact of its "daunting scenes" which I deem to be complete failure to attract its audience. The worst even trampled me, cause my friend failed to come on time at the theater because she was busy assisting her boyfriend in looking for an appropriate lodge to stay in for one night. I wasn't really disappointed with that matter, but this movie is a matter indeed for me, poor plot, useless storyline, naively created and I don't know what to say anymore.<br /><br />The title doesn't suggest anyway the creeps and horror it failed to overture us viewers, maybe the beating of the animals could get more the creeps if they show it in theaters the real situational play. Good luck to anyone who attempts to watch it anyway.

Negative Review 2
*spoiler alert!* it just

Analyze Average Review Length

In [13]:
train_df['review_length'] = train_df['review'].apply(lambda x: len(x.split()))

print("Average review length:")
print(train_df['review_length'].mean())

Average review length:
231.36275


NLP Pipeline

Download NLTK Resources

In [15]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [16]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [17]:
stop_words = set(stopwords.words('english'))

In [20]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [21]:
def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Join words again
    text = " ".join(tokens)

    return text

In [22]:
train_df['clean_text'] = train_df['review'].apply(preprocess_text)

test_df['clean_text'] = test_df['review'].apply(preprocess_text)

TF-IDF Vectorization

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train = tfidf.fit_transform(train_df['clean_text'])
X_test = tfidf.transform(test_df['clean_text'])

y_train = train_df['label']
y_test = test_df['label']

Train Logistic Regression

In [24]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

LogisticRegression()

Predictions

In [25]:
y_pred = model.predict(X_test)

Accuracy

In [26]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8888


Evaluation

In [27]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.88      0.89      5000
           1       0.88      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



Confusion Matrix

In [33]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test, y_pred)

cm_table = pd.DataFrame(
    cm,
    index=['Actual Negative', 'Actual Positive'],
    columns=['Predicted Negative', 'Predicted Positive']
)

print("The confusion Matrix :")
print(cm_table)

The confusion Matrix :
                 Predicted Negative  Predicted Positive
Actual Negative                4401                 599
Actual Positive                 513                4487


## **Transformer-Based Sentiment Analysis**

In [34]:
!pip install transformers torch

Load pretrained transformer pipeline

In [36]:
from transformers import pipeline

bert_sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Use a smaller test sample

In [37]:
test_sample = test_df.sample(500, random_state=42)

texts = test_sample['review'].tolist()
y_true = test_sample['label'].tolist()

Run sentiment prediction

In [38]:
predictions = bert_sentiment(
    texts,
    truncation=True,
    max_length=512
)

Convert predictions to labels

In [39]:
y_pred_bert = []

for pred in predictions:
    if pred['label'] == 'POSITIVE':
        y_pred_bert.append(1)
    else:
        y_pred_bert.append(0)

Evaluation

In [40]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

bert_accuracy = accuracy_score(y_true, y_pred_bert)

print("Transformer Accuracy:", bert_accuracy)

print(classification_report(y_true, y_pred_bert))

Transformer Accuracy: 0.898
              precision    recall  f1-score   support

           0       0.87      0.93      0.90       251
           1       0.93      0.86      0.89       249

    accuracy                           0.90       500
   macro avg       0.90      0.90      0.90       500
weighted avg       0.90      0.90      0.90       500



Confusion Matrix as Table

In [41]:
cm_bert = confusion_matrix(y_true, y_pred_bert)

cm_bert_table = pd.DataFrame(
    cm_bert,
    index=['Actual Negative', 'Actual Positive'],
    columns=['Predicted Negative', 'Predicted Positive']
)

print(cm_bert_table)

                 Predicted Negative  Predicted Positive
Actual Negative                 234                  17
Actual Positive                  34                 215


## **Compare Traditional vs Transformer**

In [42]:
comparison = pd.DataFrame({
    'Model': ['Traditional NLP + Logistic Regression', 'Transformer Model'],
    'Accuracy': [accuracy, bert_accuracy],
    'Feature Method': ['TF-IDF', 'Transformer embeddings'],
    'Speed': ['Fast', 'Slower'],
    'Context Understanding': ['Limited', 'Better']
})

comparison

,Model,Accuracy,Feature Method,Speed,Context Understanding
0,Traditional NLP + Logistic Regression,0.8888,TF-IDF,Fast,Limited
1,Transformer Model,0.8980,Transformer embeddings,Slower,Better


The traditional NLP model using TF-IDF and Logistic Regression achieved an accuracy of 88.88%, while the transformer-based model achieved 89.80% accuracy.

The transformer model showed slightly better performance because it uses contextual embeddings and understands semantic relationships between words more effectively than TF-IDF.

However, the traditional NLP model was faster and computationally simpler, making it suitable for lightweight applications. In contrast, the transformer model required more computation time but provided better contextual understanding and improved handling of nuanced language.

Overall, transformer-based models offer better performance for sentiment analysis tasks, while traditional NLP methods remain efficient and practical for simpler applications.